In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
import uuid

In [7]:
model = ChatOllama(model="mistral")

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("system", "{summary}"),
    ("human","{input}")
])

In [76]:
messages = {}

In [ ]:
summary_prompt = ChatPromptTemplate.from_template("""
Current conversation summary:

{summary}

New messages:

{messages}

Update the summary.

Keep:
- User preferences
- Important facts
- Ongoing tasks
- Decisions already made
- Don't miss anything everything should be covered, every message should be covered
- use long context ,
over all summary of chats

Return only the updated summary.
""")

In [68]:
summary_chain = (
    {
        "summary":lambda x: x["summary"],
        "messages": lambda x: x["history"]
    }|
    summary_prompt | model)

In [69]:
chain = (
    {
        "history": lambda x: x["history"],
        "summary": lambda x: x["summary"],
        "input": lambda x: x["input"]
    }|  
    prompt | model)

In [70]:
def get_session_id(session_id):
    if session_id not in messages:
        messages[session_id] = {"messages_count":0,"messages":[],"summary":"No summary yet."}
    return messages[session_id]

In [71]:
message_len = 10
last_message_len = 4

In [ ]:
def chat(session_id,user_input):
    history  = get_session_id(session_id)
    history["messages"].append(HumanMessage(content=user_input))
    history["messages_count"] += 1
    load_summary = history["summary"]
    load_history = history["messages"][-last_message_len:]

    response = chain.invoke({
        "history": load_history,
        "summary": load_summary,
        "input": user_input
    })

    history["messages"].append(AIMessage(content=response.content))

    if history["messages_count"] % message_len == 0:
        history["summary"] = summary_chain.invoke({
            "history": history["messages"],
            "summary": load_summary
        })

        history["messages_count"] = message_len - last_message_len
    
    return response.content





In [73]:
user1 = str(uuid.uuid4())#ram
user2 = str(uuid.uuid4())#john

In [74]:
print("user1:",user1)
print("user2:",user2)

user1: 8767f758-65ac-4e6d-b042-add41f05b693
user2: 5e80cb1e-4c8a-420f-98b2-c6f6f6317b7d


In [82]:
while True:
    user_input = input("Enter the message, exit to break:")

    if user_input.lower() == "exit":
        break

    ai = chat(user1,user_input)
    print("AI Response :",ai)
    print(60*"*")

AI Response :  That's fantastic news! Winning a hackathon is an excellent achievement that showcases your skills and dedication in the field of machine learning and programming. It can open up opportunities for internships, job offers, and networking with like-minded individuals. Congratulations on your win! If you have any questions or need guidance on how to build upon this accomplishment or prepare for future competitions, feel free to ask.
************************************************************
AI Response :  It seems that you shared your interest in machine learning and mentioned winning a hackathon. To help you continue developing your skills and pursue further opportunities, I'd recommend checking out the following resources tailored to the UK:

1. Courses from The Open University offer various degree programs and short courses on topics such as computer science, data analysis, and artificial intelligence.
2. The Alan Turing Institute hosts a free summer school for undergra

In [81]:
messages

{'8767f758-65ac-4e6d-b042-add41f05b693': {'messages_count': 9,
  'messages': [HumanMessage(content='hi i am RAM', additional_kwargs={}, response_metadata={}),
   AIMessage(content=' Hello, RAM! How can I assist you today? Let me help you with your questions or tasks to the best of my ability. Is there something specific you would like to know or discuss?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
   HumanMessage(content='i like chicken dum biryani ', additional_kwargs={}, response_metadata={}),
   AIMessage(content=" I'm glad to hear that you enjoy Chicken Dum Biryani! It is a delicious and flavorful dish. If you would like some recipes for Chicken Dum Biryani, I can certainly provide you with a few options. Here are two simple yet tasty recipes:\n\n1. Hyderabadi Chicken Dum Biryani:\nIngredients:\n- 2 cups basmati rice\n- 1 kg chicken (preferably chicken pieces with bones)\n- 6 large onions, thinly sliced\n- 5 tablespoons ghee or oil\n- 2 teas

In [83]:
messages

{'8767f758-65ac-4e6d-b042-add41f05b693': {'messages_count': 9,
  'messages': [HumanMessage(content='hi i am RAM', additional_kwargs={}, response_metadata={}),
   AIMessage(content=' Hello, RAM! How can I assist you today? Let me help you with your questions or tasks to the best of my ability. Is there something specific you would like to know or discuss?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
   HumanMessage(content='i like chicken dum biryani ', additional_kwargs={}, response_metadata={}),
   AIMessage(content=" I'm glad to hear that you enjoy Chicken Dum Biryani! It is a delicious and flavorful dish. If you would like some recipes for Chicken Dum Biryani, I can certainly provide you with a few options. Here are two simple yet tasty recipes:\n\n1. Hyderabadi Chicken Dum Biryani:\nIngredients:\n- 2 cups basmati rice\n- 1 kg chicken (preferably chicken pieces with bones)\n- 6 large onions, thinly sliced\n- 5 tablespoons ghee or oil\n- 2 teas